Import Libraries

In [4]:
# Supply Chain Logistics Optimization

## Data Preprocessing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


 Load Dataset

In [5]:
import pandas as pd

data = pd.read_excel(
    "../data/Supply chain logistics problem.xlsx",
    sheet_name=None
)

print("Data loaded successfully.")

Data loaded successfully.


In [6]:
data.keys()

dict_keys(['OrderList', 'FreightRates', 'WhCosts', 'WhCapacities', 'ProductsPerPlant', 'VmiCustomers', 'PlantPorts'])

Assign each sheet to a DataFrame

In [7]:
orders = data["OrderList"]
freight = data["FreightRates"]
plant_ports = data["PlantPorts"]
products_plant = data["ProductsPerPlant"]
vmi_customers = data["VmiCustomers"]
wh_capacities = data["WhCapacities"]
wh_costs = data["WhCosts"]
orders.head()


,Order ID,Order Date,Origin Port,Carrier,TPT,Service Level,Ship ahead day count,Ship Late Day count,Customer,Product ID,Plant Code,Destination Port,Unit quantity,Weight
0,1.447296e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,808,14.30
1,1.447158e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,3188,87.94
2,1.447139e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,2331,61.20
3,1.447364e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,847,16.16
4,1.447364e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,2163,52.34


In [8]:
datasets = {
    "Orders": orders,
    "Freight": freight,
    "Plant Ports": plant_ports,
    "Products per Plant": products_plant,
    "VMI Customers": vmi_customers,
    "Warehouse Capacities": wh_capacities,
    "Warehouse Costs": wh_costs
}

for name, df in datasets.items():
    print(name, ":", df.shape)

Orders : (9215, 14)
Freight : (1540, 11)
Plant Ports : (22, 2)
Products per Plant : (2036, 2)
VMI Customers : (14, 2)
Warehouse Capacities : (19, 2)
Warehouse Costs : (19, 2)


In [9]:
for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print(df.isnull().sum())


Orders
Order ID                0
Order Date              0
Origin Port             0
Carrier                 0
TPT                     0
Service Level           0
Ship ahead day count    0
Ship Late Day count     0
Customer                0
Product ID              0
Plant Code              0
Destination Port        0
Unit quantity           0
Weight                  0
dtype: int64

Freight
Carrier         0
orig_port_cd    0
dest_port_cd    0
minm_wgh_qty    0
max_wgh_qty     0
svc_cd          0
minimum cost    0
rate            0
mode_dsc        0
tpt_day_cnt     0
Carrier type    0
dtype: int64

Plant Ports
Plant Code    0
Port          0
dtype: int64

Products per Plant
Plant Code    0
Product ID    0
dtype: int64

VMI Customers
Plant Code    0
Customers     0
dtype: int64

Warehouse Capacities
Plant ID           0
Daily Capacity     0
dtype: int64

Warehouse Costs
WH           0
Cost/unit    0
dtype: int64


### Missing Value Check

All seven dataset tables were checked for missing values. No missing values were identified in any column. Therefore, no missing-value imputation or row removal is required at this stage.

In [10]:
print("Orders")
print("Unique Products :", orders["Product ID"].nunique())
print("Unique Customers:", orders["Customer"].nunique())
print("Unique Plants   :", orders["Plant Code"].nunique())
print("Unique Ports    :", orders["Origin Port"].nunique())
print("Unique Carriers :", orders["Carrier"].nunique())

Orders
Unique Products : 772
Unique Customers: 46
Unique Plants   : 7
Unique Ports    : 3
Unique Carriers : 3


In [11]:
product_plant_count = (
    products_plant
    .groupby("Product ID")["Plant Code"]
    .nunique()
)

print(product_plant_count.describe())

count    1540.000000
mean        1.322078
std         0.812608
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         5.000000
Name: Plant Code, dtype: float64


In [12]:
product_plant_count.value_counts().sort_index()

Plant Code
1    1271
2     141
3      43
4      71
5      14
Name: count, dtype: int64

### Product–Plant Feasibility

The ProductsPerPlant table shows that products are not universally available at all plants. Most products are associated with a single feasible plant, while some products can be supplied by multiple plants. Therefore, product–plant compatibility must be considered as a constraint in the optimization model.

This relationship will be used as common input for both the Simulated Annealing and MILP approaches.

In [13]:
plant_port_count = (
    plant_ports
    .groupby("Plant Code")["Port"]
    .nunique()
)

print(plant_port_count)

Plant Code
PLANT01    2
PLANT02    1
PLANT03    1
PLANT04    1
PLANT05    1
PLANT06    1
PLANT07    2
PLANT08    1
PLANT09    1
PLANT10    2
PLANT11    1
PLANT12    1
PLANT13    1
PLANT14    1
PLANT15    1
PLANT16    1
PLANT17    1
PLANT18    1
PLANT19    1
Name: Port, dtype: int64


### Plant–Port Connectivity

The PlantPorts table defines the ports through which each plant can ship products. Most plants are connected to one port, while PLANT01, PLANT07, and PLANT10 are connected to two ports.

Therefore, plant–port connectivity is included as a feasibility condition when generating transportation options. The same processed relationship will be used by both the Simulated Annealing and MILP approaches.

In [14]:
print(wh_capacities)

   Plant ID  Daily Capacity 
0   PLANT15               11
1   PLANT17                8
2   PLANT18              111
3   PLANT05              385
4   PLANT02              138
5   PLANT01             1070
6   PLANT06               49
7   PLANT10              118
8   PLANT07              265
9   PLANT14              549
10  PLANT16              457
11  PLANT12              209
12  PLANT11              332
13  PLANT09               11
14  PLANT03             1013
15  PLANT13              490
16  PLANT19                7
17  PLANT08               14
18  PLANT04              554


### Plant Capacity

The WhCapacities table provides the daily capacity associated with each plant. The dataset contains 19 plants with different capacity levels.

Plant capacity will be incorporated as a constraint in the optimization model. The exact capacity measure used in the model will be determined based on the meaning of the capacity field and its relationship with order quantities and weights.

The same capacity data and interpretation will be used by both the Simulated Annealing and MILP approaches.

In [15]:
print(wh_costs)


         WH  Cost/unit
0   PLANT15   1.415063
1   PLANT17   0.428947
2   PLANT18   2.036254
3   PLANT05   0.488144
4   PLANT02   0.477504
5   PLANT01   0.566976
6   PLANT06   0.554088
7   PLANT10   0.493582
8   PLANT07   0.371424
9   PLANT14   0.634330
10  PLANT16   1.919808
11  PLANT12   0.773132
12  PLANT11   0.555247
13  PLANT09   0.465071
14  PLANT03   0.517502
15  PLANT13   0.469707
16  PLANT19   0.639763
17  PLANT08   0.522857
18  PLANT04   0.428503


### Data Cleaning – Freight Feasibility

During data preprocessing, orders were checked against the available freight-rate information and other operational constraints.
- Total orders in the original dataset: 9,215
- Orders with valid freight and operational options: 6,438
- Orders without a valid freight option: 2,777
The 2,777 orders were excluded from the optimization dataset because no valid freight-rate option was available for them.
The 6,438 valid orders are used for both Simulated Annealing (SA) and MILP to ensure a fair comparison.

In [16]:
# ============================================================
# Data Cleaning and Feasibility Filtering
# ============================================================

# Check whether an order has a valid freight option
def valid_freight(order):

    # Special case: V44_3 + CRF has no freight-rate record
    # but is treated as a valid zero-freight case.
    if order["Carrier"] == "V44_3" and order["Service Level"] == "CRF":
        return True

    # Find a matching freight-rate record
    match = freight[
        (freight["Carrier"] == order["Carrier"]) &
        (freight["orig_port_cd"] == order["Origin Port"]) &
        (freight["dest_port_cd"] == order["Destination Port"]) &
        (freight["svc_cd"] == order["Service Level"]) &
        (freight["minm_wgh_qty"] <= order["Weight"]) &
        (freight["max_wgh_qty"] >= order["Weight"])
    ]

    return not match.empty


# ------------------------------------------------------------
# Step 1: Remove orders without a valid freight option
# ------------------------------------------------------------

valid_freight_mask = orders.apply(valid_freight, axis=1)

freight_clean_orders = orders[
    valid_freight_mask
].copy()

print("Original orders:", len(orders))
print("After freight filtering:", len(freight_clean_orders))
print(
    "Removed due to freight:",
    len(orders) - len(freight_clean_orders)
)


# ============================================================
# Step 2: Apply operational feasibility constraints
# ============================================================

# Product -> compatible plants
product_to_plants = (
    products_plant
    .groupby("Product ID")["Plant Code"]
    .apply(list)
    .to_dict()
)

# Plant -> connected ports
plant_to_ports = (
    plant_ports
    .groupby("Plant Code")["Port"]
    .apply(list)
    .to_dict()
)

# Customer -> allowed VMI plants
vmi_customer_to_plants = {}

for _, row in vmi_customers.iterrows():

    plant = row["Plant Code"]

    for customer in str(row["Customers"]).split(","):

        customer = customer.strip()

        if customer:
            vmi_customer_to_plants.setdefault(
                customer, []
            ).append(plant)


# Check whether an order has at least one feasible plant
def valid_plant_assignment(order):

    product = order["Product ID"]
    customer = order["Customer"]
    origin_port = order["Origin Port"]

    # Plants that can produce the product
    candidate_plants = product_to_plants.get(
        product, []
    )

    for plant in candidate_plants:

        # Plant must be connected to the origin port
        if origin_port not in plant_to_ports.get(
            plant, []
        ):
            continue

        # Apply VMI restriction
        if customer in vmi_customer_to_plants:

            allowed_plants = vmi_customer_to_plants[
                customer
            ]

            if plant not in allowed_plants:
                continue

        # A valid plant assignment exists
        return True

    # No feasible plant found
    return False


# Check all freight-valid orders
valid_plant_mask = freight_clean_orders.apply(
    valid_plant_assignment,
    axis=1
)

clean_orders = freight_clean_orders[
    valid_plant_mask
].copy()


# ============================================================
# Final cleaning results
# ============================================================

print("\nFinal cleaning results")
print("----------------------")
print("Original orders:", len(orders))
print(
    "After freight filtering:",
    len(freight_clean_orders)
)
print(
    "After operational feasibility filtering:",
    len(clean_orders)
)
print(
    "Total excluded orders:",
    len(orders) - len(clean_orders)
)

Original orders: 9215
After freight filtering: 7845
Removed due to freight: 1370

Final cleaning results
----------------------
Original orders: 9215
After freight filtering: 7845
After operational feasibility filtering: 6438
Total excluded orders: 2777


In [18]:
# Create a cleaned workbook with all original sheets
# but replace OrderList with the 7,845 valid orders

cleaned_file = "../data/cleaned_supply_chain.xlsx"

with pd.ExcelWriter(cleaned_file, engine="openpyxl") as writer:
    clean_orders.to_excel(writer, sheet_name="OrderList", index=False)
    freight.to_excel(writer, sheet_name="FreightRates", index=False)
    wh_costs.to_excel(writer, sheet_name="WhCosts", index=False)
    wh_capacities.to_excel(writer, sheet_name="WhCapacities", index=False)
    products_plant.to_excel(writer, sheet_name="ProductsPerPlant", index=False)
    vmi_customers.to_excel(writer, sheet_name="VmiCustomers", index=False)
    plant_ports.to_excel(writer, sheet_name="PlantPorts", index=False)

print("Cleaned workbook created successfully.")
print("Orders in cleaned workbook:", len(clean_orders))

Cleaned workbook created successfully.
Orders in cleaned workbook: 6438


In [19]:
cleaned_data = pd.read_excel(
    "../data/cleaned_supply_chain.xlsx",
    sheet_name=None
)

print(cleaned_data.keys())

dict_keys(['OrderList', 'FreightRates', 'WhCosts', 'WhCapacities', 'ProductsPerPlant', 'VmiCustomers', 'PlantPorts'])
